# Library

In [65]:
import os
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../')))

import numpy as np
import matplotlib.pyplot as plt
import cv2
import glob
import torch
import pytorch_lightning as pl
import torch.nn as nn
import wandb
import torch.optim as optim
from dotenv import load_dotenv
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from torchvision.datasets import DatasetFolder
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from PIL import Image
from collections import Counter
from tqdm import tqdm
from sklearn.metrics import confusion_matrix
from torch.utils.data import random_split, Subset
import gc
from sklearn.utils.class_weight import compute_class_weight
from pytorch_lightning.loggers import WandbLogger
import seaborn as sns
import pandas as pd

from datetime import datetime

# Loading data

In [2]:
saved_dir = "04-10-2025/npz"

train = np.load(f"{saved_dir}/train_data.npz")
valid = np.load(f"{saved_dir}/val_data.npz")
test  = np.load(f"{saved_dir}/test_data.npz")

In [ ]:
x_train = train['x_train']
y_train_keys = train['y_train']

x_valid = valid['x_val'] 
y_valid_keys = valid['y_val']

x_test = test['x_test']
y_test_keys = test['y_test'] # label with mapping source
y_test = np.array([key.split('_')[0] for key in y_test_keys]) # remove the mapping source from the label

In [7]:
np.unique(y_train_keys)

array(['PB1 Basmati', 'Pusa', 'Pusa 1121 Basmati', 'Pusa 1401 Basmati',
       'Pusa 1509 Basmati', 'Pusa 1718 Basmati', 'Sharbati Rice',
       'Sugandha Basmati', 'TAJ'], dtype='<U17')

In [5]:
Counter(y_train_keys)

Counter({np.str_('Pusa 1509 Basmati'): 48814,
         np.str_('Pusa 1121 Basmati'): 44993,
         np.str_('Pusa 1401 Basmati'): 29734,
         np.str_('Pusa 1718 Basmati'): 27123,
         np.str_('Sharbati Rice'): 22188,
         np.str_('Sugandha Basmati'): 17079,
         np.str_('TAJ'): 12021,
         np.str_('PB1 Basmati'): 10751,
         np.str_('Pusa'): 9727})

In [12]:
main_classes = ["Pusa 1509 Basmati", "Pusa 1121 Basmati", "Pusa 1401 Basmati", "Pusa 1718 Basmati"]
# Transform labels to a trinary classification problem
def transform_breed(breed):
    return "Other" if breed not in main_classes else breed
y_train = np.array([transform_breed(key) for key in y_train_keys])
y_valid = np.array([transform_breed(key) for key in y_valid_keys])
y_test = np.array([transform_breed(key) for key in y_test ])

In [13]:
Counter(y_train), Counter(y_valid), Counter(y_test)

(Counter({np.str_('Other'): 71766,
          np.str_('Pusa 1509 Basmati'): 48814,
          np.str_('Pusa 1121 Basmati'): 44993,
          np.str_('Pusa 1401 Basmati'): 29734,
          np.str_('Pusa 1718 Basmati'): 27123}),
 Counter({np.str_('Other'): 9548,
          np.str_('Pusa 1509 Basmati'): 6128,
          np.str_('Pusa 1121 Basmati'): 4687,
          np.str_('Pusa 1401 Basmati'): 3867,
          np.str_('Pusa 1718 Basmati'): 3839}),
 Counter({np.str_('Other'): 7976,
          np.str_('Pusa 1121 Basmati'): 6758,
          np.str_('Pusa 1509 Basmati'): 5071,
          np.str_('Pusa 1401 Basmati'): 3422,
          np.str_('Pusa 1718 Basmati'): 2152}))

In [14]:
# Encode labels using Keras utilities
unique_classes = np.unique(y_train) # extract distinct classes
class_indices = {label: index for index, label in enumerate(unique_classes)} # class with index
y_train_encoded = np.array([class_indices[label] for label in y_train]) # [0, 1, 2, 3, 0, 1, 2, 3, ...]
y_valid_encoded = np.array([class_indices[label] for label in y_valid]) # [0, 1, 2, 3, 0, 1, 2, 3, ...]
y_test_encoded = np.array([class_indices[label] for label in y_test]) # [0, 1, 2, 3, 0, 1, 2, 3, ...]

# Calculate class weights
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.unique(y_train_encoded),
    y=y_train_encoded
)
class_weight_dict = {index: weight for index, weight in enumerate(class_weights)}

In [15]:
y_train_onehot = np.zeros((len(y_train_encoded), len(unique_classes)), dtype=int)
y_valid_onehot = np.zeros((len(y_valid_encoded), len(unique_classes)), dtype=int)
y_test_onehot = np.zeros((len(y_test_encoded), len(unique_classes)), dtype=int)

In [ ]:
# One-hot encode the labels
def one_hot_encode(y_onehot, y_encoded):
    for i, label in enumerate(y_encoded):
        y_onehot[i, label] = 1
    
    return y_onehot[:,1:] # Exclude the first column (index 0) to match the number of classes (Exclude "Other" class for now)

y_train_onehot = one_hot_encode(y_train_onehot, y_train_encoded)
y_valid_onehot = one_hot_encode(y_valid_onehot, y_valid_encoded)
y_test_onehot = one_hot_encode(y_test_onehot, y_test_encoded)

In [17]:
y_train_encoded[1000], y_train_onehot[1000], y_train_encoded[1], y_train_onehot[1], y_train_encoded[2], y_train_onehot[2], y_train_encoded[3], y_train_onehot[3]

(np.int64(0),
 array([0, 0, 0, 0]),
 np.int64(1),
 array([1, 0, 0, 0]),
 np.int64(1),
 array([1, 0, 0, 0]),
 np.int64(1),
 array([1, 0, 0, 0]))

In [18]:
class_indices

{np.str_('Other'): 0,
 np.str_('Pusa 1121 Basmati'): 1,
 np.str_('Pusa 1401 Basmati'): 2,
 np.str_('Pusa 1509 Basmati'): 3,
 np.str_('Pusa 1718 Basmati'): 4}

In [19]:
intensity_ranges = {
        "brightness": np.linspace(0.9, 1.1, 5),  # Range should be >0
        "contrast": np.linspace(0.9, 1.1, 5),
        "saturation": np.linspace(0.9, 1.1, 5),  # Ensure positive values
        "hue": np.linspace(-0.05, 0.05, 5)  # Hue range should be (-0.5, 0.5)
        }

# transform = transforms.Compose([
#     transforms.Resize((224, 224)),  # Resize the image
#     # transforms.RandomApply(
#     # [transforms.ColorJitter(brightness=(intensity_ranges["brightness"].min(), intensity_ranges["brightness"].max()),
#     #                         contrast=(intensity_ranges["contrast"].min(), intensity_ranges["contrast"].max()),
#     #                         saturation=(intensity_ranges["saturation"].min(), intensity_ranges["saturation"].max()),
#     #                         hue=(intensity_ranges["hue"].min(), intensity_ranges["hue"].max()))],
#     #                         p=0.8,),
#     # transforms.ToTensor(),  # Convert image to tensor
#     torch.tensor(dtype=torch.float32).permute(2, 0, 1) / 255.0,
#     transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]), # Normalize image
# ])

# Transform if needed (example: normalize and convert to tensor)
class CustomTransform:
    def __call__(self, image):
        # Custom transformation logic here (e.g., normalize, augment, etc.)
        image = torch.tensor(image, dtype=torch.float32).permute(2, 0, 1) / 255.0
        return image

In [20]:
class CustomDataset(Dataset):
    def __init__(self, x_data, y_data, transform=None):
        self.x_data = x_data  # NumPy array of images embeddings
        self.y_data = y_data  # NumPy array of labels
        self.transform = transform

    def __len__(self):
        return len(self.x_data)

    def __getitem__(self, idx):

        img = self.x_data[idx]   
        if self.transform:
            img = self.transform(img)    
        label = self.y_data[idx]  # Get corresponding label

        return img, torch.tensor(label, dtype=torch.float)

In [21]:
transform = CustomTransform()
train_dataset = CustomDataset(x_train, y_train_onehot, transform)
valid_dataset = CustomDataset(x_valid, y_valid_onehot, transform)
test_dataset = CustomDataset(x_test, y_test_onehot, transform)

In [22]:
train_dataset[1000][1]

tensor([0., 0., 0., 0.])

In [23]:
train_dataset[0][1].shape

torch.Size([4])

In [24]:
y_train_encoded.shape

(222430,)

In [25]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(valid_dataset, batch_size=64)
test_loader = DataLoader(test_dataset, batch_size=64)

In [26]:
for x, y in train_loader:
    print(x.shape, y.shape)
    # print(y)
    break

torch.Size([64, 3, 224, 224]) torch.Size([64, 4])


In [27]:
idx_to_class = {v:k for k, v in class_indices.items()}

# Model initializing

In [28]:
class Classifier(pl.LightningModule):
    def __init__(self, num_classes=4, class_weights=None):
        super(Classifier, self).__init__()
        self.base_model = models.densenet121(pretrained=True)
        self.base_model.classifier = nn.Sequential(
                                    nn.Linear(self.base_model.classifier.in_features, 128),
                                    nn.ReLU(),
                                    nn.Dropout(0.5),
                                    nn.Linear(128, 64),
                                    nn.ReLU(),
                                    nn.BatchNorm1d(64),
                                    nn.Dropout(0.5),
                                    nn.Linear(64, num_classes),
                                    nn.Sigmoid()
                    )
        self.model = self.base_model
        self.criterion = nn.BCELoss()

    def forward(self, x):
        return self.model(x)

    def training_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.criterion(y_hat, y)
        
        # Get predicted labels (taking the class with the highest probability)
        preds = y_hat > 0.5  # Convert probabilities to binary predictions (0 or 1)
    
        # # Calculate accuracy: Number of correct predictions / Total number of predictions
        class_wise_correct = (preds == y).sum(dim=0)
        class_wise_accuracy = class_wise_correct / y.size(0)  # Divide by batch size to get accuracy

        avg_accuracy = torch.mean(class_wise_accuracy)

        for i in range(len(class_wise_accuracy)):
            self.log(f"train_acc_{idx_to_class[i+1]}", class_wise_accuracy[i], prog_bar=True, on_step=False, on_epoch=True)

        self.log("train_loss", loss, prog_bar=True, on_step=False,on_epoch=True)
        self.log("avg_train_acc", avg_accuracy, prog_bar=True, on_step=False, on_epoch=True)
        return loss

    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        loss = self.criterion(y_hat, y)
        # Get predicted labels (taking the class with the highest probability)
        preds = y_hat > 0.5  # Convert probabilities to binary predictions (0 or 1)
    
        # # Calculate accuracy: Number of correct predictions / Total number of predictions
        class_wise_correct = (preds == y).sum(dim=0)
        class_wise_accuracy = class_wise_correct / y.size(0)  # Divide by batch size to get accuracy

        avg_accuracy = torch.mean(class_wise_accuracy)

        for i in range(len(class_wise_accuracy)):
            self.log(f"val_acc_{idx_to_class[i+1]}", class_wise_accuracy[i], prog_bar=True, on_step=False, on_epoch=True)


        avg_accuracy = torch.mean(class_wise_accuracy)
        self.log("val_loss", loss, prog_bar=True, on_step=False, on_epoch=True)
        self.log("avg_val_acc", avg_accuracy, prog_bar=True, on_step=False, on_epoch=True)

    def configure_optimizers(self):
        optimizer = optim.Adam(self.parameters(), lr=1e-4)
        return optimizer

# Model training

In [29]:
# Get current date & time in "DD-MM_HH-MM" format
timestamp = datetime.now().strftime("%d-%m_%H-%M")
timestamp

'11-04_09-43'

In [30]:
dirpath = "checkpoints/10April/"
os.makedirs(dirpath, exist_ok=True)
filename = f"best_model_{timestamp}-{{epoch:02d}}-{{avg_val_acc:.4f}}-trinaryCNN"

In [53]:
model = Classifier(num_classes=len(class_indices)-1)

/home/easyrice/anaconda3/envs/torch/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/easyrice/anaconda3/envs/torch/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DenseNet121_Weights.IMAGENET1K_V1`. You can also use `weights=DenseNet121_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [54]:
# class FreezeUnfreezeCallback(pl.Callback):
#     def on_train_epoch_start(self, trainer, pl_module):
#         current_epoch = trainer.current_epoch

#         if (current_epoch // 3) % 2 == 0:
#             # Freeze feature extractor
#             for param in pl_module.model.features.parameters():
#                 param.requires_grad = False
#             print(f"[Epoch {current_epoch}] Freezing feature extractor")
#         else:
#             # Unfreeze
#             for param in pl_module.model.features.parameters():
#                 param.requires_grad = True
#             print(f"[Epoch {current_epoch}] Unfreezing feature extractor")

#         # Update optimizer
#         new_optimizer = optim.Adam(
#             filter(lambda p: p.requires_grad, pl_module.parameters()),
#             lr=1e-3 if (current_epoch // 3) % 2 == 0 else 1e-4
#         )
#         trainer.optimizers = [new_optimizer]

In [ ]:
# freeze_callback = FreezeUnfreezeCallback()
# Define the checkpoint callback
checkpoint_callback = ModelCheckpoint(
    monitor='avg_val_acc',         # Monitor validation accuracy
    dirpath=dirpath,    # Directory to save the checkpoints
    filename=filename,     # Filename for the best model
    save_top_k=1,              # Save only the best model
    mode='max',                # We want the maximum validation accuracy
)
early_stop_callback = EarlyStopping(monitor="avg_val_acc", min_delta=0.00, patience=10, verbose=False, mode="max")
load_dotenv()
wandb.login(key=os.getenv("WANDB_API_KEY"))
wandb_logger = WandbLogger(project="MPIndiaCNN",  name=f"CNN ({timestamp}) (5class Indian Rice)")

# Trainer setup
trainer = pl.Trainer(logger=wandb_logger,
                     max_epochs=100, 
                     accelerator="gpu" if torch.cuda.is_available() else "mps",
                     callbacks=[checkpoint_callback, early_stop_callback])
trainer.fit(model, train_loader, val_loader)

In [ ]:
wandb.finish()

# Evaluation

In [31]:
def predict_and_transform(model, val_loader):
    
    predictions = []
    

    # Disable gradient computation for inference
    model.to("cuda")
    model.eval()
    with torch.no_grad():
        for batch in val_loader:
            x_batch = batch[0].to("cuda")
            x_batch = x_batch.squeeze(1)
            y_hat = model(x_batch)
            preds = (y_hat > 0.5).int() # Convert bool to int
            predictions.extend(preds.cpu().numpy())  # Move back to CPU and store the predictions

    predicted_classes = np.array(predictions)

    
    return predicted_classes

In [32]:
dirpath, filename

('checkpoints/10April/',
 'best_model_11-04_09-43-{epoch:02d}-{avg_val_acc:.4f}-trinaryCNN')

In [33]:
classifier_model = Classifier.load_from_checkpoint(f'{dirpath}best_model_10-04_14-20-epoch=19-val_acc=0.0000-trinaryCNN.ckpt', strict=False, num_classes=4)
predicted_classes = predict_and_transform(classifier_model, test_loader)

del classifier_model # Delete the model to free up memory
gc.collect()
torch.cuda.empty_cache()

/home/easyrice/anaconda3/envs/torch/lib/python3.11/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/easyrice/anaconda3/envs/torch/lib/python3.11/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=DenseNet121_Weights.IMAGENET1K_V1`. You can also use `weights=DenseNet121_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [37]:
class_indices

{np.str_('Other'): 0,
 np.str_('Pusa 1121 Basmati'): 1,
 np.str_('Pusa 1401 Basmati'): 2,
 np.str_('Pusa 1509 Basmati'): 3,
 np.str_('Pusa 1718 Basmati'): 4}

In [32]:
def plot_confusion_matrix_percentage(y_true, y_pred, class_labels):
    cm = confusion_matrix(y_true, y_pred, labels=range(len(class_labels)))
    cm_percentage = cm.astype('float') / cm.sum(axis=1, keepdims=True) * 100  # Convert to percentage
    cm_percentage = np.nan_to_num(cm_percentage)  # Handle division by zero
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm_percentage, annot=True, fmt='.2f', cmap='Blues', xticklabels=class_labels, yticklabels=class_labels)
    plt.xlabel('Predicted Labels')
    plt.ylabel('True Labels')
    plt.title('Confusion Matrix (Percentage)->DensNet121')
    plt.show()

In [33]:
class_indices

{np.str_('Other'): 0,
 np.str_('Pusa 1121 Basmati'): 1,
 np.str_('Pusa 1509 Basmati'): 2}

In [ ]:
plot_confusion_matrix_percentage(y_test_encoded, predicted_classes, ['Other', '1121', '1509'])

In [51]:
y_test_keys

array(['Pusa 1121 Basmati', 'Pusa 1121 Basmati', 'Pusa 1121 Basmati', ...,
       'Pusa 1121 Basmati', 'Pusa 1121 Basmati', 'Pusa 1121 Basmati'],
      dtype='<U17')

In [41]:
class_indices

{np.str_('Other'): 0,
 np.str_('Pusa 1121 Basmati'): 1,
 np.str_('Pusa 1401 Basmati'): 2,
 np.str_('Pusa 1509 Basmati'): 3,
 np.str_('Pusa 1718 Basmati'): 4}

In [40]:
y_test_keys, len(y_test_keys)

(array(['Pusa 1509 Basmati_000037', 'Pusa 1509 Basmati_000037',
        'Pusa 1509 Basmati_000037', ..., 'TAJ_000077', 'TAJ_000077',
        'TAJ_000077'], dtype='<U24'),
 25379)

In [42]:
unique_combinations = np.unique(predicted_classes, axis=0)
unique_combinations

array([[0, 0, 0, 0],
       [0, 0, 0, 1],
       [0, 0, 1, 0],
       [0, 1, 0, 0],
       [1, 0, 0, 0],
       [1, 1, 0, 0]], dtype=int32)

In [ ]:
onehot_to_class = {
    (0, 0, 0, 0): "other",
    (0, 0, 0, 1): "Pusa 1718 Basmati",
    (0, 0, 1, 0): "Pusa 1509 Basmati",
    (0, 1, 0, 0): "Pusa 1401 Basmati",
    (1, 0, 0, 0): "Pusa 1121 Basmati",
    (1, 1, 0, 0): "Pusa 1121 Basmati & Pusa 1401 Basmati"
}
mapped_labels = [onehot_to_class.get(tuple(row), "Unknown") for row in predicted_classes]

In [ ]:
test_df = pd.read_csv("04-10-2025/04-10-2025test5class.csv")
test_df['folder'] = test_df['folder'].str.split('/').str[-1] # Extract the folder

In [83]:
import pandas as pd

idx_to_class = {v: k for k, v in class_indices.items()}
# Convert predicted class indices to actual class names
# predicted_class_names = np.array([idx_to_class[idx] for idx in predicted_classes])
mapped_labels = [onehot_to_class.get(tuple(row), "Unknown") for row in predicted_classes]
df = pd.DataFrame({'breed': y_test,'key_image_name': y_test_keys, 'prediction': mapped_labels})

# Group by key_image_name and count prediction occurrences
result = df.groupby(['key_image_name', 'prediction']).size().unstack(fill_value=0)

In [84]:
result = result.reset_index()
result['breed'] = result['key_image_name'].map(lambda x: x.split('_')[0])
result['folder'] = result['key_image_name'].str.split('_').str[1]
merged = result.merge(test_df[['folder', 'state', 'city', 'mandi', 'mill', 'date']], on='folder', how='left')

merged

,key_image_name,Pusa 1121 Basmati,Pusa 1121 Basmati & Pusa 1401 Basmati,Pusa 1401 Basmati,Pusa 1509 Basmati,Pusa 1718 Basmati,other,breed,folder,state,city,mandi,mill,date
0,Pusa 1121 Basmati_000012,279,0,1,19,2,2,Pusa 1121 Basmati,000012,Haryana,Gharaunda,-,M/S Akash,02/10/2024
1,Pusa 1121 Basmati_000046,832,0,18,0,28,15,Pusa 1121 Basmati,000046,Haryana,-,Haryana,-,08/07/2024
2,Pusa 1121 Basmati_000046,832,0,18,0,28,15,Pusa 1121 Basmati,000046,Haryana,-,Haryana,-,08/07/2024
3,Pusa 1121 Basmati_000050,938,0,5,0,13,9,Pusa 1121 Basmati,000050,-,Sonipat,Sonipat,-,08/05/2024
4,Pusa 1121 Basmati_000050,938,0,5,0,13,9,Pusa 1121 Basmati,000050,-,Sonipat,Sonipat,-,08/05/2024
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
67,TAJ_000076,3,0,2,4,1,208,TAJ,000076,-,Panipat,Panipat,SUNRISE,14/01/2025
68,TAJ_000077,1,0,2,3,0,245,TAJ,000077,-,Samalkha,Samalkha,SUNRISE,14/01/2025
69,TAJ_000078,8,0,4,4,1,586,TAJ,000078,-,Gannaur,Gannaur,VEER,17/01/2025
70,TAJ_000078,8,0,4,4,1,586,TAJ,000078,-,Gannaur,Gannaur,VEER,17/01/2025


In [85]:
merged.to_csv("04-11-2025-intermediate_results.csv", index=False)